<a href="https://colab.research.google.com/github/alscop/ESAA-26-1/blob/main/ESAA_OB_week03_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 파이썬 머신러닝 완벽 가이드 9장 p.584~601


# 09 추천 시스템

## 9-01 추천 시스템의 개요와 배경

### 추천 시스템의 개요

Recommendations. 추천 엔진  
전자상거래 업체 ~ 콘텐츠 포털까지 추천시스템 고도화 진행중

### 온라인 스토어의 필수 요소, 추천 시스템

한정된 시간이라는 제약 속 제품 선택 어려움 겪을수록 압박 커짐.  

### 추천 시스템의 유형

2개의 방식
- 콘텐츠 기반 필터링(Content based filtering)
- 협업 필터링(Collaborative Filtering)

초창기에는 전자, 넷플 경연 대회에서 후자 우승하며 후자가 대중화됨.  
하이브리드 형식도 있음.

## 9-02 콘텐츠 기반 필터링 추천 시스템

특정한 아이템을 매우 선호하는 경우, 그 아이템과 **비슷한 콘텐츠를 가진 다른 아이템**을 추천하는 방식  

## 9-03 최근접 이웃 협업 필터링

**<협업필터링>**

아이템에 매긴 평점 정보나 상품 구매 이력과 같은 **사용자 행동 양식(User Behavior)** 만을 기반으로 추천을 수행  
친구들에게 물어보는 것과 유사한 방식  

목표)  
사용자-아이템 평점 매트릭스와 같은 축적된 사용자 행동 데이터를 기반 -> 사용자가 아직 평가하지 않은 아이템을 예측 평가(Predicted Rating)하는 것

- 최근접 이웃 방식 = 메모리 방식
  - 사용자 기반(User—User): 사용자 간 유사도 이용
  - **아이템 기반(Item-Item)**: 아이템의 평가 척도(선호도, 평점)가 유사한 아이템을 추천 / ~아이템간의 속성~.
- 잠재 요인 방식

둘 다 사용자-아이템 평점 매트릭스 사용함.  
- 행: 개별 사용자
- 열: 개별 아이템
- 값: 평점

형태 유지 필수!  
만약 로우 레벨형태라면 pandas `pivot_table()` 함수로 형태 변경해야함.

- 다차원 행렬(열(아이템)이 많음)  
- 희소 행렬

-> 텍스트 분석처럼, 유사도 측정에 코사인 유사도 이용.




## 9-04 잠재 요인 협업 필터링

### 잠재 요인 협업 필터링의 이해

사용자-아이템 평점 매트릭스 속 잠재 요인을 추출해 사용자가 아직 평가하지 않은 아이템에 대한 예측 평가 수행.  
(잠재요인 정의는 어려우나...  )

**행렬 분해**: 대규모 다차원 행렬을 SVD와 같은 차원 감소 기법으로 분해

[다차원 희소 행렬]
- 사용자-아이템 행렬 데이터  

-> (잠재요인 추출)

[저차원 밀집 행렬]
- 사용자-잠재요인 행렬
- 잠재 요인-아이템 행렬

**분해된 두 행렬 내적곱**  
-> NEW 예측 사용자—아이템 평점 행렬 데이터 생성  
-> 예측 평점을 생성


### 행렬 분해의 이해

다차원 -> 저차원 행렬로 분해.  
- SVD(Singular Vector Decomposition): L2 규제 이용 & NaN 없는 행렬에만 적용 가능. 이때
  - 확률적 경사 하강법(Stochastic Gradient Descent, SGD)
  - ALS(Alternating Least Squares)
- NMF(Non—Negative Matrix Factorization)

분해 = 인수분해.  
R(MXN) => P(MXK) * Q'T(KXN)  

내적을 통해 미정 값을 포함한 모든 평점 값을 예측할 수 있음.


### 확률적 경사 하강법을 이용한 행렬 분해

P와 Q 행렬로 계산된 예측 R 행렬 값이  
실제 R 행렬 값과 가장 최소의 오류를 가질 수 있도록  
반복적인 비용 함수 최적화를 통해  
P와 Q를 유추해내는 것


1. P, Q를 임의 값의 행렬로 설정
2. 예측 R(P*Q]T) 계산 후 실제 R과 비교, 오류값 계산
3. **오류값 최소화할 수 있도록 P, Q값 적절히 업데이트**
4. 만족할 오류값 가질 때까지 2-3번 반복



>  SGD를 이용해 행렬 분해를 수행하는 예제

In [2]:
import numpy as np

# 원본 행렬 R 생성, 분해 행렬 P와 Q 초기화, 잠재 요인 차원 K는 3으로 설정.
R = np.array([[4, np.nan, np.nan, 2, np.nan],
              [np.nan, 5, np.nan, 3, 1],
              [np.nan, np.nan, 3, 4, 4],
              [5, 2, 1, 2, np.nan]])
num_users, num_items = R.shape
K = 3

# P와 Q 행렬의 크기를 지정하고 정규 분포를 가진 임의의 값으로 입력
np.random.seed(1)
P = np.random.normal(scale=1./K, size=(num_users, K))
Q = np.random.normal(scale=1./K, size=(num_items, K))

In [3]:
from sklearn.metrics import mean_squared_error

# 실제 R 행렬과 예측 행렬의 오차를 구하는 get_rmse( ) 함수 생성
def get_rmse(R, P, Q, non_zeros):
  error = 0
  # 두 개의 분해된 행렬 P와 Q.T의 내적으로 예측 R 행렬 생성
  full_pred_matrix = np.dot(P, Q.T)

  # 실제 R 행렬에서 널이 아닌 값의 위치 인덱스 추출해 실제 R 행렬과 예측 행렬의 RMSE 추출
  x_non_zero_ind = [non_zero[0] for non_zero in non_zeros]
  y_non_zero_ind = [non_zero[1] for non_zero in non_zeros]
  R_non_zeros = R[x_non_zero_ind, y_non_zero_ind]
  full_pred_matrix_non_zeros = full_pred_matrix[x_non_zero_ind, y_non_zero_ind]
  mse = mean_squared_error(R_non_zeros, full_pred_matrix_non_zeros)
  rmse = np.sqrt(mse)

  return rmse

In [5]:
# R > 0인 행 위치, 열 위치, 값을 non_zeros 리스트에 저장.
non_zeros = [ (i, j, R[i,j]) for i in range(num_users) for j in range(num_items) if R[i, j] > 0 ]

steps = 1000    # 1000번 반복
learning_rate = 0.01    # SGD의 학습률
r_lambda = 0.01     # L2 규제 계수

# SGD 기법으로 P, Q 매트릭스를 계속 업데이트.
for step in range(steps):
  for i, j, r in non_zeros:
    # 실제 값과 예측 값의 차이인 오류 값 구함.
    eij = r - np.dot(P[i,:], Q[j,:].T)
    # 규제를 반영한 SGD 업데이트 공식 적용
    P[i, :] = P[i, :] + learning_rate*(eij * Q[j, :] - r_lambda*P[i, :])
    Q[i, :] = Q[i, :] + learning_rate*(eij * P[i, :] - r_lambda*Q[j, :])

    rmse = get_rmse(R, P, Q, non_zeros)
    if (step % 50) == 0:
      print('### iteration step :', step, 'rmse :', rmse)

### iteration step : 0 rmse : 3.261355059488935
### iteration step : 0 rmse : 3.2605214069579556
### iteration step : 0 rmse : 3.254094793105207
### iteration step : 0 rmse : 3.2500869277626547
### iteration step : 0 rmse : 3.2486738087972027
### iteration step : 0 rmse : 3.2474548877950076
### iteration step : 0 rmse : 3.24518044920153
### iteration step : 0 rmse : 3.2434058182832457
### iteration step : 0 rmse : 3.2391581485676153
### iteration step : 0 rmse : 3.238187528869786
### iteration step : 0 rmse : 3.237829346615862
### iteration step : 0 rmse : 3.2365448868302784
### iteration step : 50 rmse : 1.8478536324657573
### iteration step : 50 rmse : 1.8484971624126865
### iteration step : 50 rmse : 1.8484033742198744
### iteration step : 50 rmse : 1.8481410628196289
### iteration step : 50 rmse : 1.847711801713254
### iteration step : 50 rmse : 1.840777225953471
### iteration step : 50 rmse : 1.8429709523406095
### iteration step : 50 rmse : 1.8508266897572723
### iteration step :

In [6]:
# 분해된 P, Q 함수를 이용해 예측행렬 만들어 출력
pred_matrix = np.dot(P, Q.T)
print('예측 행렬: \n', np.round(pred_matrix, 3))

예측 행렬: 
 [[  3.969 -17.204  16.879   2.011   0.496]
 [ -0.413   4.97   -1.872   3.044   0.075]
 [  0.892   0.247   3.106   4.226   0.271]
 [  0.316  -0.13   -1.4    -0.745   0.072]]


널 아닌 값들은 꽤 비슷  
널인 값들은 새로운 예측값으로 채워짐  